# Ascend C Matmul 算子——跑通第一个矩阵乘法程序

通过上一节的学习，我们已经了解了 Matmul 算子的基本概念和重要性。本节是 Matmul 算子开发的基础实践章节，我们将基于 CANN Samples 中的 `matmul/main.asc` 示例代码，**逐段解析并跑通一个完整的矩阵乘法程序**。

本节学习大纲如下：
- 环境准备
- Ascend C 核函数编写基础（复习）
- Matmul 核函数逐段解析
- Host 侧主函数实现
- 编译运行与结果验证

matmul 采用的是 **Cube 编程模型**：数据经过 GM → L1 → L0A/L0B → L0C 四级存储层级，通过 M-MAD（Matrix Multiply-Add）指令在 CubeCore 上完成矩阵乘累加计算。

In [ ]:
!mkdir -p Sources
!mkdir -p ../third_party
!git clone https://gitcode.com/cann/ops-tensor.git ../third_party/ops-tensor
!cd ../third_party/ops-tensor && git checkout f2e975775ca754c04084dece850dd6cc76bfbc5b
import os
import subprocess

result = subprocess.run(
    ['bash', '-l', '-c', 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'],
    capture_output=True, text=True, check=True
)
for line in result.stdout.strip().split('\n'):
    line = line.strip()
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value
print("\n🎉 Environment initialization process completed successfully!")

## 1. 如何编写核函数

核函数（Kernel Function）是 Ascend C 算子设备侧的入口。Ascend C 允许用户使用核函数这种 C/C++ 函数的语法扩展来管理设备侧的运行代码。

Matmul 的核函数声明与 HelloWorld 示例形式相同，但多了模板参数和 GM_ADDR 入参：

```cpp
template <typename T>
__global__ __aicore__ void MatmulKernel(GM_ADDR aGm, GM_ADDR bGm, GM_ADDR cGm,
                                       uint32_t m, uint32_t k, uint32_t n)
```

### 修饰符说明
- **\_\_global__**：表明该函数是可以被主机端（Host）调用的核函数。
- **\_\_aicore__**：指定该核函数在昇腾 NPU 处理器的 AI Core 单元上执行。

## 2. Ascend C 异构并行编程模型

### 2.1 Host-Device 异构协同机制
Ascend C 异构计算架构分为 Host 侧和 Device 侧（Device 侧对应昇腾 AI 处理器），两者协同完成计算任务：
- **Host 侧**：负责运行时管理——ACL 初始化、设备内存分配、数据搬运（Host↔Device）、核函数启动、资源释放。
- **Device 侧**：执行开发者编写的 Kernel 核函数，在 AI Core 上完成矩阵运算等计算密集型任务。

### 2.2 SPMD 并行计算
Ascend C 算子编程是 SPMD（Single-Program Multiple-Data）编程——"一份代码，多处执行，处理的数据不同"。
具体到 Matmul 算子中：将 M×N 输出矩阵划分为多个 Tile，每个 AI Core 处理若干个 Tile（通过 `GetBlockIdx()` 识别自己的身份），实现多核并行。

```cpp
// 多核任务分配：每个核以 blockNum 为步长处理属于自己的 Tile
for (uint64_t tileIdx = curBlockIdx; tileIdx < tileNum; tileIdx += blockNum) {
    // curBlockIdx = GetBlockIdx()，blockNum = GetBlockNum()
    uint64_t mTileIdx = tileIdx / nTileNum;
    uint64_t nTileIdx = tileIdx % nTileNum;
    // 处理当前 (mTileIdx, nTileIdx) 对应的 Tile ...
}
```

## 3. 如何调用核函数

核函数使用内核调用符 `<<<...>>>` 来规定执行配置：

```cpp
matmul::MatmulKernel<bfloat16_t><<<numBlocks, nullptr, stream>>>(
    deviceInput, deviceWeight, deviceOutput, m, k, n);
```

参数说明：
- **numBlocks**：核函数将在几个 AI Core 上执行，通过 `PlatformAscendCManager::GetCoreNumAic()` 获取可用 AI Core 数量。
- **nullptr**：l2ctrl 保留参数，设置为 nullptr。
- **stream**：类型为 `aclrtStream`，任务队列，用于管理任务的并行。
- **模板参数 `<bfloat16_t>`**：指定设备侧计算使用的数据类型。

一个完整的核函数调用流程包括：
1. ACL 初始化（`aclInit`）
2. 申请运行管理资源（device、stream）
3. 申请 Host 和 Device 侧内存，数据拷贝到 Device
4. 调用 `<<<...>>>` 启动核函数
5. 同步等待（`aclrtSynchronizeStream`）
6. 数据拷回 Host 侧，资源释放
7. ACL 去初始化（`aclFinalize`）

## 4. 跑通第一个 Matmul 程序

### 4.1 编写 Matmul 程序

下面我们逐步编写 `Sources/matmul.asc`，它包含核函数实现（Device 侧）和主函数实现（Host 侧）两部分。
完整代码来源于 CANN Samples 中 `0_Introduction/matmul/main.asc`。

#### 4.1.1 头文件引入与前置声明

代码开头需要引入 Host 侧和 Device 侧所需的头文件，并声明工具函数和常量的前置声明。

In [ ]:
%%writefile Sources/matmul.asc
/*
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * Implementation of matrix multiplication kernel for Ascend processors
 */

#include <iostream>
#include <vector>
#include <cmath>
#include <memory>
#include <random>
#include <iomanip>
#include <algorithm>
#include <cstring>
#include <cstdint>

#include "acl/acl.h"
#include "kernel_basic_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "include/tensor_api/tensor.h"

namespace tool {
constexpr static int32_t L0C_C0 = 16;
constexpr static uint16_t ZERO_FLAG = 0;
__aicore__ inline uint64_t CeilDiv(uint64_t a, uint64_t b);
float Bf16ToFloat(uint16_t h);
uint16_t FloatToBf16(float f);
template <typename T>
void FillRandomData(std::vector<T>& data, T min, T max);
template <typename T>
void ComputeGolden(int m, int k, int n, std::vector<T>& hostInput,
                   std::vector<T>& hostWeight, std::vector<T>& goldenOutput);
template <typename T>
std::vector<uint64_t> Compare(std::vector<T>& hostOutput, std::vector<T>& goldenOutput);
} // namespace tool

**代码说明：**

- **`acl/acl.h`**：Ascend Computing Language 头文件，提供 Host 侧运行管理 API（`aclInit`、`aclrtMalloc`、`aclrtMemcpy` 等）。
- **`kernel_basic_intf.h`**：提供 Ascend C 核函数基础接口（`GetBlockIdx`、`GetBlockNum`、`SetFlag`、`WaitFlag` 等）。
- **`tiling/platform/platform_ascendc.h`**：提供平台信息管理，如获取 AI Core 数量。
- **`include/tensor_api/tensor.h`**：提供 Te（Tensor Engine）API——`MakeTensor`、`MakeFrameLayout`、`MakeMemPtr`、`Copy`、`Mmad` 等。
- **`namespace tool { ... }`**：声明工具常量和函数的前置声明：
  - `L0C_C0 = 16`：L0C Buffer 的对齐原子值
  - `ZERO_FLAG = 0`：硬件同步标志值
  - `CeilDiv`：向上取整除法
  - `Bf16ToFloat` / `FloatToBf16`：bfloat16 ↔ float32 转换
  - `FillRandomData`：随机数据填充
  - `ComputeGolden`：CPU 侧参考矩阵乘法
  - `Compare`：精度对比验证

#### 4.1.2 L1 布局类型别名

L1 上矩阵 A 和 B 的布局根据是否转置自动选择 NZ 或 ZN 的内存排布，通过编译期模板元编程 `conditional_t` 实现。

In [ ]:
%%writefile -a Sources/matmul.asc

namespace AscendC::Te {
static constexpr bool transA = false;
static constexpr bool transB = false;
template <typename T>
using MakeLayoutAL1 =
    AscendC::Std::conditional_t<transA,
        AscendC::Te::FrameLayoutFormat<AscendC::Te::ZNLayoutPtn, AscendC::Te::LayoutTraitDefault<T>>,
        AscendC::Te::FrameLayoutFormat<AscendC::Te::NZLayoutPtn, AscendC::Te::LayoutTraitDefault<T>>>;
template <typename T>
using MakeLayoutBL1 =
    AscendC::Std::conditional_t<transB,
        AscendC::Te::FrameLayoutFormat<AscendC::Te::ZNLayoutPtn, AscendC::Te::LayoutTraitDefault<T>>,
        AscendC::Te::FrameLayoutFormat<AscendC::Te::NZLayoutPtn, AscendC::Te::LayoutTraitDefault<T>>>;
} // namespace AscendC::Te

**代码说明：**

- **`transA` / `transB`**：控制矩阵 A 和 B 是否在 L1 上转置的编译期标志。当前均为 `false`（不转置），因此 L1 上 A 和 B 都使用 **NZ** 布局。
- **`MakeLayoutAL1<T>`**：根据 `transA` 的值，编译期自动选择对应的 `FrameLayoutFormat` 类型——若 `transA=true` 则为 ZN 布局，否则为 NZ 布局。
- **`LayoutTraitDefault<T>`**：告诉编译器元素类型 T 的字节大小和 C0 对齐原子值（如 `bfloat16_t` → C0=16）。

#### 4.1.3 核函数实现

核函数 `MatmulKernel` 是整个程序的**Device 侧核心**，实现矩阵乘法的分块（Tiling）计算。下面先给出完整代码，再分段解析其关键设计。

In [ ]:
%%writefile -a Sources/matmul.asc

namespace matmul {
template <typename T>
__global__ __aicore__ void MatmulKernel(GM_ADDR aGm, GM_ADDR bGm, GM_ADDR cGm, uint32_t m, uint32_t k, uint32_t n)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIC_ONLY);

    // === Tiling 参数 ===
    uint64_t baseM = 256;
    uint64_t baseN = 256;
    uint64_t baseK = 256 / sizeof(T);
    uint64_t kL1 = 1024 / sizeof(T);
    uint64_t mTileNum = tool::CeilDiv(m, baseM);
    uint64_t nTileNum = tool::CeilDiv(n, baseN);
    uint64_t tileNum = mTileNum * nTileNum;
    uint64_t kL1TileNum = tool::CeilDiv(k, kL1);
    uint64_t tailKL1 = k - (kL1TileNum - 1) * kL1;
    uint64_t tailBaseM = m - (mTileNum - 1) * baseM;
    uint64_t tailBaseN = n - (nTileNum - 1) * baseN;
    uint64_t l0cOffset = 0;

    uint64_t curBlockIdx = AscendC::GetBlockIdx();
    uint64_t blockNum = AscendC::GetBlockNum();

    // === 构建 GM 张量（ND 布局） ===
    auto layoutA = AscendC::Te::MakeFrameLayout<AscendC::Te::NDExtLayoutPtn>(m, k);
    auto layoutB = AscendC::Te::MakeFrameLayout<AscendC::Te::NDExtLayoutPtn>(k, n);
    auto layoutC = AscendC::Te::MakeFrameLayout<AscendC::Te::NDExtLayoutPtn>(m, n);

    auto tensorAgm = AscendC::Te::MakeTensor(
        AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(reinterpret_cast<__gm__ T*>(aGm)), layoutA);
    auto tensorBgm = AscendC::Te::MakeTensor(
        AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(reinterpret_cast<__gm__ T*>(bGm)), layoutB);
    auto tensorCgm = AscendC::Te::MakeTensor(
        AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(reinterpret_cast<__gm__ T*>(cGm)), layoutC);

    // === 初始化同步标志 ===
    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(tool::ZERO_FLAG);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(tool::ZERO_FLAG);
    AscendC::SetFlag<AscendC::HardEvent::FIX_M>(tool::ZERO_FLAG);

    // === 多核 Tile 处理 ===
    for (uint64_t tileIdx = curBlockIdx; tileIdx < tileNum; tileIdx += blockNum) {
        uint64_t mTileIdx = tileIdx / nTileNum;
        uint64_t nTileIdx = tileIdx % nTileNum;
        int64_t curM = mTileIdx == (mTileNum - 1) ? tailBaseM : baseM;
        int64_t curN = nTileIdx == (nTileNum - 1) ? tailBaseN : baseN;

        // 从 GM 切出当前 Tile
        auto tensorAGmBlock = tensorAgm.Slice(
            AscendC::Te::MakeCoord(mTileIdx * baseM, 0L), AscendC::Te::MakeShape(curM, k));
        auto tensorBGmBlock = tensorBgm.Slice(
            AscendC::Te::MakeCoord(0L, nTileIdx * baseN), AscendC::Te::MakeShape(k, curN));
        auto tensorCGmBlock = tensorCgm.Slice(
            AscendC::Te::MakeCoord(mTileIdx * baseM, nTileIdx * baseN), AscendC::Te::MakeShape(curM, curN));
        auto layoutL0C = AscendC::Te::MakeFrameLayout<AscendC::Te::NZLayoutPtn, AscendC::Std::Int<tool::L0C_C0>>(curM, curN);
        auto tensorL0C = AscendC::Te::MakeTensor(
            AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0C, float>(l0cOffset), layoutL0C);

        AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(tool::ZERO_FLAG);
        // === K 维度 L1 循环 ===
        for (uint64_t iter0 = 0; iter0 < kL1TileNum; ++iter0) {
            AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(tool::ZERO_FLAG);

            auto curGmBKL1 = (iter0 + 1 == kL1TileNum) ? (k - iter0 * kL1) : kL1;
            auto curGmAKL1 = curGmBKL1;

            uint64_t l1BufferAOffset = 0;
            uint64_t l1BufferBOffset = baseM * kL1 * sizeof(T);

            // GM -> L1 搬运
            auto copyGM2L1 = AscendC::Te::MakeCopy(AscendC::Te::CopyGM2L1{});
            auto layoutAL1 = AscendC::Te::MakeLayoutAL1<T>{}(curM, curGmAKL1);
            auto tensorAL1 = AscendC::Te::MakeTensor(
                AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, T>(l1BufferAOffset), layoutAL1);
            auto tensorAGmTile = tensorAGmBlock.Slice(
                AscendC::Te::MakeCoord(0, iter0 * kL1), AscendC::Te::MakeShape(curM, curGmAKL1));
            AscendC::Te::Copy(copyGM2L1, tensorAL1, tensorAGmTile);

            auto layoutBL1 = AscendC::Te::MakeLayoutBL1<T>{}(curGmBKL1, curN);
            auto tensorBL1 = AscendC::Te::MakeTensor(
                AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, T>(l1BufferBOffset), layoutBL1);
            auto tensorBGmTile = tensorBGmBlock.Slice(
                AscendC::Te::MakeCoord(iter0 * kL1, 0), AscendC::Te::MakeShape(curGmBKL1, curN));
            AscendC::Te::Copy(copyGM2L1, tensorBL1, tensorBGmTile);

            AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(tool::ZERO_FLAG);
            AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(tool::ZERO_FLAG);

            // === K 维度 L0 循环（乒乓） ===
            uint64_t kL0IterNum = tool::CeilDiv(curGmBKL1, baseK);
            uint64_t tailKL0 = curGmBKL1 - (kL0IterNum - 1) * baseK;
            for (uint16_t iter1 = 0; iter1 < kL0IterNum; ++iter1) {
                AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(tool::ZERO_FLAG);

                uint64_t curKL0 = (iter1 + 1 == kL0IterNum) ? tailKL0 : baseK;

                // L1 -> L0A / L0B 搬运
                auto copyL12L0A = AscendC::Te::MakeCopy(AscendC::Te::CopyL12L0A{});
                auto copyL12L0B = AscendC::Te::MakeCopy(AscendC::Te::CopyL12L0B{});
                auto layoutAL0 = AscendC::Te::MakeFrameLayout<
                    AscendC::Te::NZLayoutPtn, AscendC::Te::LayoutTraitDefault<T>>(curM, curKL0);
                auto tensorAL0 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0A, T>(0), layoutAL0);
                auto tensorAL1Tile = tensorAL1.Slice(
                    AscendC::Te::MakeCoord(0, iter1 * baseK), AscendC::Te::MakeShape(curM, curKL0));
                AscendC::Te::Copy(copyL12L0A, tensorAL0, tensorAL1Tile);

                auto layoutBL0 = AscendC::Te::MakeFrameLayout<
                    AscendC::Te::ZNLayoutPtn, AscendC::Te::LayoutTraitDefault<T>>(curKL0, curN);
                auto tensorBL0 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0B, T>(0), layoutBL0);
                auto tensorBL1Tile = tensorBL1.Slice(
                    AscendC::Te::MakeCoord(iter1 * baseK, 0), AscendC::Te::MakeShape(curKL0, curN));
                AscendC::Te::Copy(copyL12L0B, tensorBL0, tensorBL1Tile);

                AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(tool::ZERO_FLAG);
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(tool::ZERO_FLAG);

                // M-MAD 矩阵乘累加
                AscendC::Te::MmadParams para;
                para.cmatrixInitVal = (iter1 == 0 && iter0 == 0);
                para.m = curM;
                para.n = curN;
                para.k = curKL0;

                auto MadOp = AscendC::Te::MakeMmad(
                    AscendC::Te::MmadOperation{}, AscendC::Te::MmadTraitDefault{});
                AscendC::Te::Mmad(MadOp, tensorL0C, tensorAL0, tensorBL0, para);

                AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(tool::ZERO_FLAG);
            }
            AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(tool::ZERO_FLAG);
        }
        AscendC::SetFlag<AscendC::HardEvent::M_FIX>(tool::ZERO_FLAG);
        AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(tool::ZERO_FLAG);

        // L0C -> GM 结果写回
        auto copyL0C2GM = AscendC::Te::MakeCopy(AscendC::Te::CopyL0C2GM{});
        AscendC::Te::Copy(copyL0C2GM, tensorCGmBlock, tensorL0C);

        AscendC::SetFlag<AscendC::HardEvent::FIX_M>(tool::ZERO_FLAG);
    }
    // 最终收尾同步
    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(tool::ZERO_FLAG);
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(tool::ZERO_FLAG);
    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(tool::ZERO_FLAG);
}
} // namespace matmul

**核函数关键设计解析：**

##### 1）任务类型声明
```cpp
KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIC_ONLY);
```
告诉编译器：该核函数必须在 **AI Cube Core** 上运行。

##### 2）三维分块（Tiling）策略

| 维度 | 变量 | 默认大小 | 含义 |
|------|------|----------|------|
| M, N | `baseM`, `baseN` | 256 | 输出矩阵在 M/N 方向的单次计算块 |
| K (L1级) | `kL1` | 1024 / sizeof(T) | K 维度在 L1 上的单次搬运块 |
| K (L0级) | `baseK` | 256 / sizeof(T) | K 维度在 L0 上的单次计算块 |

通过 `CeilDiv` 计算每个维度的 Tile 数量，并处理不能被整除时的尾块（`tailBaseM`、`tailBaseN`、`tailKL1`、`tailKL0`）。

##### 3）Tensor 构建三部曲

Cube 编程中，任何可操作的张量都需经过三步构建：
```
MakeMemPtr<Location>(地址)  ->  "数据在哪"（物理存储体 + 指针/偏移）
MakeFrameLayout<Ptn>(形状)   ->  "数据怎么排"（NZ/ZN/NDExt 排列方式）
MakeTensor(memPtr, layout)  ->  粘合为可被 Slice/Copy/Mmad 操作的张量
```

##### 4）存储层级与布局要求

```
GM (NDExtLayout)  ->  CopyGM2L1  ->  L1 (NZ/ZN)
                                ->  CopyL12L0A  ->  L0A (NZ)
                                ->  CopyL12L0B  ->  L0B (ZN)
                                ->  M-MAD 计算   ->  L0C (NZ, C0=16, float)
                                ->  CopyL0C2GM   ->  GM (结果写回)
```
L0A 使用 NZ、L0B 使用 ZN 是 **CubeCore M-MAD 的硬件要求**。L0C 固定以 `float` 精度累加，防止多次累加过程中的精度损失。

##### 5）M-MAD 指令

```cpp
AscendC::Te::MmadParams para;
para.cmatrixInitVal = (iter1 == 0 && iter0 == 0);  // 第一个 K 片段初始化 C 为 0
para.m = curM; para.n = curN; para.k = curKL0;
auto MadOp = AscendC::Te::MakeMmad(MmadOperation{}, MmadTraitDefault{});
AscendC::Te::Mmad(MadOp, tensorL0C, tensorAL0, tensorBL0, para);
```
`cmatrixInitVal` 控制 M-MAD 行为：首个 `(iter0=0, iter1=0)` 为 `true`，将 C 初始化为 0 再写入积；后续迭代为 `false`，将新积**累加**到已有部分积上，最终得到完整矩阵乘法结果 `C = sum(A_k * B_k)`。

#### 4.1.4 Host 侧辅助代码

Host 侧包含错误检查宏、命令行参数解析和帮助信息。

In [ ]:
%%writefile -a Sources/matmul.asc

#define CHECK_COND(cond, message, return_expr)              \
    do {                                                    \
        if (!(cond)) {                                      \
            std::cerr << "ERROR: " << message << std::endl; \
            return_expr;                                    \
        }                                                   \
    } while (0)

void printUsage(const std::string& programName)
{
    std::cerr << "Usage: " << programName << " m k n" << std::endl;
    std::cerr << "Args: " << std::endl;
    std::cerr << "  m: row of matrix A" << std::endl;
    std::cerr << "  k: col of matrix A" << std::endl;
    std::cerr << "  n: col of matrix B" << std::endl;
    std::cerr << "Example: " << programName << " 100 50 200" << std::endl;
}

void parseArguments(int argc, char* argv[], int& m, int& k, int& n)
{
    if (argc >= 2 && (std::string(argv[1]) == "--help" || std::string(argv[1]) == "-h")) {
        printUsage(argv[0]);
        exit(1);
    }
    if (argc < 4) {
        throw std::invalid_argument("ERROR: Lacks Arguments");
    }
    try {
        m = std::stoi(argv[1]);
        k = std::stoi(argv[2]);
        n = std::stoi(argv[3]);
    } catch (const std::invalid_argument& e) {
        throw std::invalid_argument("ERROR: m k n must be Integer");
    }
    if (m <= 0 || k <= 0 || n <= 0) {
        throw std::invalid_argument("ERROR: m k n must be positive");
    }
}

**代码说明：**

- **`CHECK_COND`**：封装条件检查逻辑的宏。若 ACL API 调用失败（返回值非 `ACL_SUCCESS`），打印错误信息并执行 `return_expr`（通常为 `return 1`）退出。配合 RAII 内存管理，确保异常退出时资源不泄漏。
- **`printUsage`**：打印命令行帮助信息，说明程序接收三个参数 `m k n`。
- **`parseArguments`**：解析并校验命令行参数——支持 `--help`/`-h` 查看帮助；验证参数个数、类型（必须为整数）和范围（必须为正数）。

#### 4.1.5 Host 侧主函数

主函数 `main` 负责 Host 侧的完整流程编排——从 ACL 初始化到核函数启动，再到精度验证和资源回收。

In [ ]:
%%writefile -a Sources/matmul.asc

int main(int argc, char* argv[])
{
    using namespace tool;
    int m, k, n;
    try {
        parseArguments(argc, argv, m, k, n);
    } catch (const std::exception& e) {
        std::cerr << e.what() << std::endl;
        printUsage(argv[0]);
        return 1;
    }

    // === 1. 初始化 ACL 资源 ===
    int32_t deviceId = 0;
    aclrtStream stream;
    auto ret = aclInit(nullptr);
    CHECK_COND(ret == ACL_SUCCESS, "aclInit failed.", return 1);
    ret = aclrtSetDevice(deviceId);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtSetDevice failed.", return 1);
    ret = aclrtCreateStream(&stream);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtCreateStream failed.", return 1);

    // === 2. Host 侧内存分配与随机数据填充 ===
    std::vector<float> hostInput(m * k, 0);
    std::vector<float> hostWeight(k * n, 0);
    std::vector<float> hostOutput(m * n, 0);
    std::vector<float> goldenOutput(m * n, 0);
    FillRandomData<float>(hostInput, -2.0f, 2.0f);
    FillRandomData<float>(hostWeight, -2.0f, 2.0f);

    // === 3. float32 -> bfloat16 转换 ===
    auto toBf16 = [](const std::vector<float>& src, std::vector<uint16_t>& dst) {
        std::transform(src.begin(), src.end(), dst.begin(), FloatToBf16);
    };
    std::vector<uint16_t> hostInputBf16(m * k, 0);
    std::vector<uint16_t> hostWeightBf16(k * n, 0);
    std::vector<uint16_t> hostOutputBf16(m * n, 0);
    std::vector<uint16_t> goldenOutputBf16(m * n, 0);
    toBf16(hostInput, hostInputBf16);
    toBf16(hostWeight, hostWeightBf16);
    toBf16(hostOutput, hostOutputBf16);
    toBf16(goldenOutput, goldenOutputBf16);

    // === 4. Device 侧内存分配（RAII 管理） ===
    auto sizeInput = hostInputBf16.size() * sizeof(uint16_t);
    auto sizeWeight = hostWeightBf16.size() * sizeof(uint16_t);
    auto sizeOutput = hostOutputBf16.size() * sizeof(uint16_t);
    void* deviceInputRaw = nullptr;
    void* deviceWeightRaw = nullptr;
    void* deviceOutputRaw = nullptr;
    ret = aclrtMalloc(&deviceInputRaw, sizeInput, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtMalloc deviceInput failed.", return 1);
    std::unique_ptr<void, aclError (*)(void*)> deviceInputPtr(deviceInputRaw, aclrtFree);
    ret = aclrtMalloc(&deviceWeightRaw, sizeWeight, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtMalloc deviceWeight failed.", return 1);
    std::unique_ptr<void, aclError (*)(void*)> deviceWeightPtr(deviceWeightRaw, aclrtFree);
    ret = aclrtMalloc(&deviceOutputRaw, sizeOutput, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtMalloc deviceOutput failed.", return 1);
    std::unique_ptr<void, aclError (*)(void*)> deviceOutputPtr(deviceOutputRaw, aclrtFree);

    uint8_t* deviceInput = reinterpret_cast<uint8_t*>(deviceInputRaw);
    uint8_t* deviceWeight = reinterpret_cast<uint8_t*>(deviceWeightRaw);
    uint8_t* deviceOutput = reinterpret_cast<uint8_t*>(deviceOutputRaw);

    // === 5. Host -> Device 数据拷贝 ===
    ret = aclrtMemcpy(deviceInput, sizeInput, hostInputBf16.data(), sizeInput, ACL_MEMCPY_HOST_TO_DEVICE);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtMemcpy deviceInput failed.", return 1);
    ret = aclrtMemcpy(deviceWeight, sizeWeight, hostWeightBf16.data(), sizeWeight, ACL_MEMCPY_HOST_TO_DEVICE);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtMemcpy deviceWeight failed.", return 1);

    // === 6. 获取 AI Core 数量并启动核函数 ===
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    CHECK_COND(ascendcPlatform != nullptr, "get ascendcPlatform failed.", return 1);
    uint32_t numBlocks = ascendcPlatform->GetCoreNumAic();
    matmul::MatmulKernel<bfloat16_t><<<numBlocks, nullptr, stream>>>(
        deviceInput, deviceWeight, deviceOutput, m, k, n);

    // === 7. 同步等待核函数执行完成 ===
    ret = aclrtSynchronizeStream(stream);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtSynchronizeStream failed.", return 1);

    // === 8. Device -> Host 结果回拷 ===
    ret = aclrtMemcpy(hostOutputBf16.data(), sizeOutput, deviceOutput, sizeOutput, ACL_MEMCPY_DEVICE_TO_HOST);
    CHECK_COND(ret == ACL_SUCCESS, "aclrtMemcpy deviceOutput failed.", return 1);

    // === 9. 精度验证 ===
    auto toFloat = [](const std::vector<uint16_t>& src, std::vector<float>& dst) {
        std::transform(src.begin(), src.end(), dst.begin(), Bf16ToFloat);
    };
    std::vector<float> hostInputFloat(m * k, 0);
    std::vector<float> hostWeightFloat(k * n, 0);
    std::vector<float> hostOutputFloat(m * n, 0);
    std::vector<float> goldenOutputFloat(m * n, 0);
    toFloat(hostInputBf16, hostInputFloat);
    toFloat(hostWeightBf16, hostWeightFloat);
    toFloat(hostOutputBf16, hostOutputFloat);
    toFloat(goldenOutputBf16, goldenOutputFloat);

    ComputeGolden<float>(m, k, n, hostInputFloat, hostWeightFloat, goldenOutputFloat);
    std::vector<uint64_t> errorIndices = Compare<float>(hostOutputFloat, goldenOutputFloat);
    if (errorIndices.size() == 0) {
        std::cout << "matmul run successfully!" << std::endl;
    } else {
        for (uint64_t i : errorIndices) {
            std::cout << "error index: " << i << ", output: " << hostOutputFloat[i]
                      << ", golden: " << goldenOutputFloat[i] << std::endl;
        }
        std::cout << "matmul run failed!" << std::endl;
    }

    // === 10. 资源释放 ===
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return 0;
}

**Host 侧主函数关键步骤解析：**

##### 1）ACL 运行时标准流程（步骤 1、7、10）
```cpp
aclInit(nullptr);                     // 初始化 ACL
aclrtSetDevice(deviceId);             // 设置当前设备
aclrtCreateStream(&stream);           // 创建任务流
// ... kernel launch ...
aclrtSynchronizeStream(stream);       // 等待核函数执行完成
aclrtDestroyStream(stream);           // 销毁流
aclrtResetDevice(deviceId);           // 复位设备
aclFinalize();                        // 去初始化
```
这是 Ascend C 程序的固有写法，以保证资源的正确获取与释放。

##### 2）bfloat16 精度转换（步骤 3）
Host 侧以 `float32` 生成随机数据并计算 CPU 参考结果，转换为 **bfloat16**（1 位符号 + 8 位指数 + 7 位尾数）后送入 Device。bfloat16 相比 float32 节省一半带宽和内存，适合 AI 推理场景。

##### 3）RAII 设备内存管理（步骤 4）
```cpp
std::unique_ptr<void, aclError (*)(void*)> deviceInputPtr(deviceInputRaw, aclrtFree);
```
使用 `unique_ptr` 配合自定义删除器 `aclrtFree`，确保设备内存在任何退出路径（正常结束或异常返回）下都能被正确释放。`ACL_MEM_MALLOC_HUGE_FIRST` 标志优先申请大页内存，提升访存带宽。

##### 4）核函数启动（步骤 6）
```cpp
uint32_t numBlocks = ascendcPlatform->GetCoreNumAic();
matmul::MatmulKernel<bfloat16_t><<<numBlocks, nullptr, stream>>>(
    deviceInput, deviceWeight, deviceOutput, m, k, n);
```
- 通过 `PlatformAscendCManager::GetCoreNumAic()` 获取 AI Core 数量
- 使用调用符 `<<<numBlocks, nullptr, stream>>>` 在所有 AI Core 上并行执行
- 模板参数 `bfloat16_t` 指定 Device 侧数据类型

##### 5）精度验证（步骤 9）
将 Device 输出与 CPU 参考结果对比，使用相对误差容忍策略（`rtol = 1/256`，约 0.39%），考虑 bfloat16 精度损失（仅 7 位尾数）。验证通过输出 `matmul run successfully!`。

#### 4.1.6 工具函数实现

最后是 `namespace tool` 中各工具函数的具体实现。

In [ ]:
%%writefile -a Sources/matmul.asc

namespace tool {
__aicore__ inline uint64_t CeilDiv(uint64_t a, uint64_t b)
{
    if (b == 0) { return a; }
    return (a + b - 1) / b;
}

template <typename T>
void FillRandomData(std::vector<T>& data, T min, T max)
{
    std::random_device rd;
    std::mt19937 gen(rd());
    if constexpr (std::is_integral<T>::value) {
        std::uniform_int_distribution<T> dist(min, max);
        for (auto& elem : data)
            elem = dist(gen);
    } else if constexpr (std::is_floating_point<T>::value) {
        std::uniform_real_distribution<T> dist(min, max);
        for (auto& elem : data)
            elem = dist(gen);
    }
}

template <typename T>
void ComputeGolden(int m, int k, int n, std::vector<T>& hostInput,
                   std::vector<T>& hostWeight, std::vector<T>& goldenOutput)
{
    for (uint32_t row = 0; row < m; ++row) {
        for (uint32_t col = 0; col < n; ++col) {
            size_t offsetGolden = row * n + col;
            T sum = 0;
            for (uint32_t iter = 0; iter < k; ++iter) {
                size_t offsetInput = row * k + iter;
                size_t offsetWeight = iter * n + col;
                sum += hostInput[offsetInput] * hostWeight[offsetWeight];
            }
            goldenOutput[offsetGolden] = sum;
        }
    }
}

template <typename T>
std::vector<uint64_t> Compare(std::vector<T>& hostOutput, std::vector<T>& goldenOutput)
{
    std::vector<uint64_t> errorIndices;
    const float rtol = 1.0f / 256;
    for (uint64_t i = 0; i < hostOutput.size(); ++i) {
        T actualValue = hostOutput[i];
        T expectValue = goldenOutput[i];
        T diff = std::fabs(actualValue - expectValue);
        if (diff > rtol * std::max(1.0f, std::fabs(expectValue))) {
            errorIndices.push_back(i);
        }
    }
    return errorIndices;
}

float Bf16ToFloat(uint16_t h)
{
    uint32_t sign = (h & 0x8000U) ? 0x80000000U : 0x00000000U;
    uint32_t exponent = (h >> 7) & 0x00FFU;
    uint32_t mantissa = h & 0x007FU;
    uint32_t f_bits = sign | (exponent << 23) | (mantissa << (23 - 7));
    return *reinterpret_cast<float*>(&f_bits);
}

uint16_t FloatToBf16(float f)
{
    uint32_t f_bits;
    std::memcpy(&f_bits, &f, sizeof(f_bits));
    return static_cast<uint16_t>(f_bits >> 16);
}
} // namespace tool

**工具函数说明：**

- **`CeilDiv(a, b)`**：向上取整除法，用于计算 Tile 数量。`(a + b - 1) / b` 实现了 `ceil(a/b)` 的整数版本。标注 `__aicore__` 表示可从 Device 侧核函数调用。
- **`FillRandomData`**：用随机数填充向量，使用 `if constexpr` 在编译期根据类型选择整数分布或浮点分布。
- **`ComputeGolden`**：标准三重循环矩阵乘法 `C[i][j] = sum A[i][k] * B[k][j]`，作为 CPU 参考基准用于精度验证。
- **`Compare`**：逐元素比较 Device 输出与参考值的相对误差。容忍度 `rtol = 1/256` 考虑了 bfloat16 的精度特性（仅 7 位尾数，约 2 位十进制有效数字）。
- **`Bf16ToFloat`**：将 bfloat16 的 16 位位模式展开为 IEEE 754 float32 格式（符号位扩展、指数位左移、尾数位低位补零）。
- **`FloatToBf16`**：取 float32 的高 16 位直接截断为 bfloat16（简单截断舍入）。

### 4.2 编译与运行

代码开发完成后，使用毕昇编译器（bisheng）将源文件编译为可执行文件，然后传入矩阵维度运行。

毕昇编译器常用选项说明：
- `--cce-aicore-arch=dav-c310-cube`：指定昇腾 AI 处理器架构版本
- `-o <file>`：指定输出文件的名称

In [ ]:
# 编译 matmul 算子
!rm -f Sources/include
!ln -sf ../../third_party/ops-tensor/include/tensor_api Sources/include
!bisheng Sources/matmul.asc --cce-aicore-arch=dav-c310-cube -o matmul_demo \
    -D_GLIBCXX_USE_CXX11_ABI=0 \
    -I Sources/include \
    -I $ASCEND_TOOLKIT_HOME/x86_64-linux/include \
    -I $ASCEND_TOOLKIT_HOME/x86_64-linux/asc/include \
    -I $ASCEND_TOOLKIT_HOME/x86_64-linux/asc/include/interface \
    -I $ASCEND_TOOLKIT_HOME/x86_64-linux/asc \
    -L $ASCEND_TOOLKIT_HOME/x86_64-linux/lib64 \
    -L $ASCEND_TOOLKIT_HOME/x86_64-linux/devlib/linux/x86_64 \
    -ltiling_api -lplatform -lm

编译成功后，运行以下命令执行矩阵乘法（以 256×256×256 为例）：

In [ ]:
# 运行: ./matmul_demo m k n
!./matmul_demo 256 256 256

如果一切正常，将输出：
```
matmul run successfully!
```

这表示 Device 侧矩阵乘法的计算结果与 CPU 参考结果在所有元素上的相对误差均在容忍范围内（`rtol = 1/256`，约 0.39%）。此外还可以尝试不同的矩阵维度：
```bash
./matmul_demo 512 512 512    # 大规模矩阵
./matmul_demo 100 50 200     # 任意维度（走尾块分支）
./matmul_demo 16 32 64       # 小矩阵
```

## 5. 总结

本节我们基于 CANN Samples 中的 `matmul/main.asc` 示例，**完整走通了一个 Ascend C Matmul 算子的开发与运行流程**。下面从编程模型、代码结构、关键技术点和开发流程四个维度进行回顾。

### 5.1 编程模型回顾

Ascend C Matmul 算子开发遵循以下核心编程模型：

| 概念 | 说明 |
|------|------|
| **Host-Device 异构** | Host 侧（CPU）负责运行时管理与任务调度；Device 侧（NPU AI Core）执行计算密集型核函数 |
| **SPMD 并行** | "一份代码，多处执行"——每个 AI Core 通过 `GetBlockIdx()` 获取身份，以 `blockNum` 为步长处理属于自己的 Tile |
| **Cube 编程** | 数据沿 **GM → L1 → L0A/L0B → L0C** 四级存储层级流动，通过 M-MAD 指令在 CubeCore 上完成矩阵乘累加 |
| **核函数调用符** | `<<<numBlocks, nullptr, stream>>>` 指定并行度和执行流 |

### 5.2 代码结构总览

整个 `matmul.asc` 源文件由以下部分构成：

```
matmul.asc
├── 头文件引入           # acl/acl.h, kernel_basic_intf.h, tensor_api/tensor.h 等
├── namespace tool       # 工具函数前置声明（CeilDiv, FillRandomData, ComputeGolden, Compare 等）
├── namespace AscendC::Te # L1 布局类型别名（MakeLayoutAL1, MakeLayoutBL1）
├── namespace matmul     # 核函数 MatmulKernel（Device 侧核心）
├── Host 侧辅助代码       # CHECK_COND 宏、printUsage、parseArguments
├── main()               # Host 侧主函数（ACL 生命周期 + 核函数启动 + 精度验证）
└── namespace tool       # 工具函数具体实现
```

### 5.3 关键技术点速查

| 技术点 | 关键代码/概念 | 位置 |
|--------|--------------|------|
| **核函数声明** | `__global__ __aicore__ void MatmulKernel(...)` | § 4.1.3 |
| **三维分块 (Tiling)** | `baseM=256, baseN=256, kL1=1024/sizeof(T), baseK=256/sizeof(T)` | § 4.1.3 |
| **Tensor 构建** | `MakeMemPtr` → `MakeFrameLayout` → `MakeTensor` 三步法 | § 4.1.3 |
| **数据搬运** | `CopyGM2L1` → `CopyL12L0A/CopyL12L0B` → `Mmad` → `CopyL0C2GM` | § 4.1.3 |
| **存储布局要求** | GM: NDExt; L1: NZ/ZN; L0A: NZ; L0B: ZN; L0C: NZ(C0=16, float) | § 4.1.3 |
| **M-MAD 指令** | `cmatrixInitVal` 控制首片段初始化 vs 后续累加 | § 4.1.3 |
| **硬件同步** | `SetFlag` / `WaitFlag` 事件对（MTE1_MTE2, M_MTE1, FIX_M 等） | § 4.1.3 |
| **bfloat16 精度** | float32 ↔ bfloat16 转换，`rtol=1/256` 容忍度 | § 4.1.5 |
| **RAII 内存管理** | `unique_ptr<void, aclError(*)(void*)>` + 自定义删除器 `aclrtFree` | § 4.1.5 |
| **ACL 生命周期** | `aclInit → SetDevice → CreateStream → ... → DestroyStream → ResetDevice → aclFinalize` | § 4.1.5 |

### 5.4 开发流程总结

完整的 Matmul 算子开发与运行流程如下图所示：

```
┌─────────────────────────────────────────────────────┐
│ 1. 环境准备                                          │
│    source set_env.sh → 克隆 ops-tensor 依赖           │
├─────────────────────────────────────────────────────┤
│ 2. 编写源文件 (Sources/matmul.asc)                   │
│    ├── 核函数: Tiling + 数据搬运 + M-MAD + 同步       │
│    ├── Host 主函数: ACL 管理 + 内存 + 启动 + 验证     │
│    └── 工具函数: 随机数/参考计算/精度对比             │
├─────────────────────────────────────────────────────┤
│ 3. 编译                                               │
│    bisheng ... --cce-aicore-arch=dav-c310-cube ...   │
├─────────────────────────────────────────────────────┤
│ 4. 运行与验证                                         │
│    ./matmul_demo m k n → matmul run successfully!    │
└─────────────────────────────────────────────────────┘
```

### 5.5 关键要点

1. **Cube 编程的核心是数据搬运与计算的流水线编排**：通过 `SetFlag`/`WaitFlag` 同步机制，实现 GM→L1、L1→L0A/L0B、M-MAD 计算、L0C→GM 写回的多级流水并行。
2. **Tiling 是支撑大矩阵计算的基础**：M/N/K 三个维度的分块策略（`baseM=256, baseN=256, baseK=256/sizeof(T)`）使任意大小的矩阵都能在有限的片上存储中完成计算，同时通过尾块处理保证正确性。
3. **Tensor API 封装了硬件细节**：`MakeMemPtr` + `MakeFrameLayout` + `MakeTensor` 的统一接口屏蔽了不同存储体（GM/L1/L0A/L0B/L0C）和不同布局（NZ/ZN/NDExt）的差异。
4. **bfloat16 是性能与精度的折中**：相比 float32 节省一半带宽和内存，7 位尾数带来的精度损失在多数 AI 场景中可接受。
5. **RAII + 错误检查宏是健壮性的保证**：`CHECK_COND` 宏在每步 ACL API 调用后进行错误检查，`unique_ptr` 配合自定义删除器确保异常退出时资源不泄漏。

通过本节的学习，你应当能够理解 Ascend C Matmul 算子的完整代码结构，并具备修改 Tiling 参数、切换数据类型、调整矩阵规模进行实验的能力。下一节将深入 Matmul 算子的性能优化技术。